# Face Mask Detector
**Two-stage pipeline:** MediaPipe (BlazeFace) for face detection → MobileNetV2 for mask classification.

| Step | Description |
|------|-------------|
| 1 | Install & import libraries |
| 2 | Download & unpack dataset |
| 3 | Explore the data |
| 4 | Build the `tf.data` pipeline |
| 5 | Build the model |
| 6 | Phase 1 — train the head (backbone frozen) |
| 7 | Phase 2 — fine-tune the backbone |
| 8 | Training curves |
| 9 | Evaluation — confusion matrix & classification report |
| 10 | Save the model |
| 11 | Gradio inference app |

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow gradio mediapipe opencv-python-headless seaborn pillow --quiet

import os
import random
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import tensorflow as tf
import mediapipe as mp
import gradio as gr
from PIL import Image

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input, Lambda
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report

# Global configuration
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print("Libraries imported successfully.")
print(f"  TensorFlow : {tf.__version__}")
print(f"  MediaPipe  : {mp.__version__}")

## Step 2 — Download & Unpack Dataset

In [ ]:
!wget -q https://github.com/chandrikadeb7/Face-Mask-Detection/archive/refs/heads/master.zip -O dataset.zip

if not os.path.exists("Face-Mask-Detection-master"):
    with zipfile.ZipFile("dataset.zip", "r") as zip_ref:
        zip_ref.extractall()

DATA_DIR = "Face-Mask-Detection-master/dataset"
print("Dataset extracted.")

## Step 3 — Explore the Data
Visualize sample images from each class and check class distribution before training.
The distribution chart tells you whether class imbalance might bias the model.

In [ ]:
class_dirs = sorted(os.listdir(DATA_DIR))
print(f"Classes found: {class_dirs}")

# Sample images grid
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Sample Images per Class", fontsize=14)

for row, class_name in enumerate(class_dirs):
    class_path = os.path.join(DATA_DIR, class_name)
    image_files = random.sample(os.listdir(class_path), 5)
    for col, fname in enumerate(image_files):
        img = Image.open(os.path.join(class_path, fname)).resize((128, 128))
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(class_name, fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

# Class distribution bar chart
counts = {cls: len(os.listdir(os.path.join(DATA_DIR, cls))) for cls in class_dirs}
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(counts.keys(), counts.values(), color=["#2ecc71", "#e74c3c"])
ax.set_title("Class Distribution")
ax.set_ylabel("Image Count")
for k, v in counts.items():
    ax.text(k, v + 10, str(v), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

print(f"Class counts: {counts}")

## Step 4 — Build the `tf.data` Pipeline
`image_dataset_from_directory` + `prefetch(AUTOTUNE)` replaces `ImageDataGenerator`.
Parallel mapping keeps the GPU fed without a CPU bottleneck.
Augmentation is applied only to the training set.

In [ ]:
raw_train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

raw_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

CLASS_NAMES = raw_train_ds.class_names
print(f"Classes: {CLASS_NAMES}")
# Note: label order matters for inference.
# CLASS_NAMES[0] -> label 0, CLASS_NAMES[1] -> label 1

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal", seed=SEED),
    tf.keras.layers.RandomRotation(0.15, seed=SEED),
    tf.keras.layers.RandomZoom(0.1, seed=SEED),
    tf.keras.layers.RandomTranslation(height_factor=0.1, width_factor=0.1, seed=SEED)
])

train_ds = raw_train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(buffer_size=tf.data.AUTOTUNE)

val_ds = raw_val_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

## Step 5 — Build the Model
MobileNetV2 backbone with a small classification head.
`Lambda(preprocess_input)` is embedded inside the model graph — the inference
function never needs to manually scale pixels, removing a common source of bugs.

In [ ]:
inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Integrated MobileNetV2 preprocessing: scales pixels to [-1, 1]
x = Lambda(preprocess_input)(inputs)

base_model = MobileNetV2(weights="imagenet", include_top=False, input_tensor=x)
base_model.trainable = False  # Freeze backbone for Phase 1

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.5, seed=SEED)(x)
outputs = Dense(1, activation="sigmoid")(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print(f"Model built. Total parameters: {model.count_params():,}")

## Step 6 — Phase 1: Train the Head (Backbone Frozen)
Only the dense layers above MobileNetV2 are updated.
Fast convergence without touching the pretrained ImageNet weights.

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1)

print("Phase 1: Training classification head (backbone frozen)...")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[early_stop],
    verbose=1
)

## Step 7 — Phase 2: Fine-Tune the Backbone (Top 30 Layers)
Unfreeze the top 30 layers of MobileNetV2 and continue training with a much
lower learning rate (`1e-5`) so pretrained weights shift gradually, not catastrophically.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

reduce_lr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7, verbose=1)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Phase 2: Fine-tuning top 30 backbone layers...")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

## Step 8 — Training Curves
Plot accuracy and loss across both phases. The dashed line marks where fine-tuning began.
Look for a gap between train and validation curves — a sign of overfitting.

In [ ]:
acc     = history1.history["accuracy"]     + history2.history["accuracy"]
val_acc = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss    = history1.history["loss"]         + history2.history["loss"]
val_los = history1.history["val_loss"]     + history2.history["val_loss"]
split   = len(history1.history["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, train_vals, val_vals, title, ylabel in zip(
    axes,
    [acc, loss],
    [val_acc, val_los],
    ["Accuracy", "Loss"],
    ["Accuracy", "Loss"]
):
    ax.plot(train_vals,  label="Train",      color="#2980b9", linewidth=2)
    ax.plot(val_vals,    label="Validation", color="#e74c3c", linewidth=2)
    ax.axvline(split - 1, color="#888", linestyle="--", label="Fine-tuning start")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.legend()

plt.suptitle("Training Curves — Both Phases", fontsize=13)
plt.tight_layout()
plt.show()

## Step 9 — Evaluation on the Validation Set
The accuracy from `model.fit` is a rough guide.
The **confusion matrix** shows *where* the model fails — which class it mixes up most.
The **classification report** gives per-class precision, recall, and F1.

In [ ]:
print("Evaluating on validation set...")
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
print(f"Validation Loss     : {val_loss:.4f}")
print(f"Validation Accuracy : {val_accuracy:.4f}")

# Collect all predictions and true labels
y_true, y_pred_prob = [], []
for images, labels in val_ds:
    probs = model.predict(images, verbose=0)
    y_pred_prob.extend(probs.flatten())
    y_true.extend(labels.numpy().flatten())

y_true      = np.array(y_true, dtype=int)
y_pred_prob = np.array(y_pred_prob)
y_pred      = (y_pred_prob > 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=ax
)
ax.set_title("Confusion Matrix")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()

# Per-class precision, recall, F1
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## Step 10 — Save the Model

In [ ]:
model.save("face_mask_detector.keras")
print("Model saved as: face_mask_detector.keras")

## Step 11 — Gradio Inference App
**Stage 1:** MediaPipe (BlazeFace) detects face bounding boxes — more robust than Haar Cascades.
**Stage 2:** Each crop is classified by MobileNetV2. `preprocess_input` is inside the model,
so raw `float32` pixel arrays are passed in directly.

> **Colab note:** `share=True` is required to get a public tunnel URL.
> Remove it if running locally.

In [ ]:
mp_face_detection = mp.solutions.face_detection

def run_inference(input_image):
    if input_image is None:
        return None, "Please upload an image."

    img_rgb = np.array(input_image)
    h, w, _ = img_rgb.shape
    output_img = img_rgb.copy()

    with mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5) as detector:
        results = detector.process(img_rgb)

    if not results.detections:
        return input_image, "No faces detected in this image."

    report_lines = []

    for idx, detection in enumerate(results.detections):
        bbox = detection.location_data.relative_bounding_box

        # Convert relative coordinates to pixel values
        x  = max(0, int(bbox.xmin * w))
        y  = max(0, int(bbox.ymin * h))
        bw = int(bbox.width  * w)
        bh = int(bbox.height * h)

        # Add 15% padding so the crop includes the full head boundary
        pad_w, pad_h = int(bw * 0.15), int(bh * 0.15)
        x1 = max(0, x - pad_w);    y1 = max(0, y - pad_h)
        x2 = min(w, x + bw + pad_w); y2 = min(h, y + bh + pad_h)

        face_crop = img_rgb[y1:y2, x1:x2]
        if face_crop.size == 0:
            continue

        # preprocess_input is embedded in the model — pass raw float32 pixels directly
        face_tensor = np.expand_dims(
            np.array(Image.fromarray(face_crop).resize((IMG_SIZE, IMG_SIZE)), dtype="float32"),
            axis=0
        )
        score = float(model.predict(face_tensor, verbose=0)[0][0])

        # CLASS_NAMES[0] = with_mask  -> score near 0
        # CLASS_NAMES[1] = without_mask -> score near 1
        no_mask    = score > 0.5
        confidence = score if no_mask else (1.0 - score)
        label      = "No Mask" if no_mask else "Mask"
        color      = (220, 50, 50) if no_mask else (34, 197, 94)

        cv2.rectangle(output_img, (x, y), (x + bw, y + bh), color, 3)
        cv2.rectangle(output_img, (x, y - 28), (x + bw, y), color, -1)
        cv2.putText(
            output_img,
            f"{label}  {confidence * 100:.1f}%",
            (x + 4, y - 8),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2, cv2.LINE_AA
        )

        report_lines.append(f"Face {idx + 1}: {label} ({confidence * 100:.1f}% confidence)")

    total = len(results.detections)
    summary = f"{total} face(s) detected.\n" + "\n".join(report_lines)
    return Image.fromarray(output_img), summary


with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("## Face Mask Detector\nUpload an image to check whether people are wearing masks.")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="pil", label="Input Image")
            run_btn     = gr.Button("Run Detection", variant="primary")
        with gr.Column():
            image_output = gr.Image(type="pil", label="Annotated Output")
            text_output  = gr.Textbox(label="Results", lines=6)

    run_btn.click(fn=run_inference, inputs=image_input, outputs=[image_output, text_output])

# share=True is required in Colab. Remove it when running locally.
app.launch(share=True, debug=False)